# Map flux
## Make the same maps as the one presented in https://arxiv.org/pdf/2104.06855.pdf

### Import modules

In [46]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import Colormap
import pandas as pd
from scipy.interpolate import interp1d

### User parameters

In [47]:
incl = 48 * np.pi / 180.0
Nx = 32
f_Ny = 0.15 # Nyquist frequency (< 0.6)
#co = sqrt(8.d0*log(2.d0))       ! difference between sigma and theta!, theta = sigma*co

### Define some constants

In [48]:
M_BH = 5.5e7 #             !Black hole mass [Msun]
R_g = 1.477e5*M_BH

h = 6.6260755e-27
k_B = 1.380658e-16
m_e = 9.10938e-28
c = 2.99792458e10
e = 4.8032068e-10
kpc = 3.0856e21  #Kpc a cm
distance = 3.8e3  #Distancia al objeto en kpc
convert = 206265e3/(distance*kpc) # image to angular size (mas)
# flux = 1.e26*(h/eV)/(4.0*np.pi*(distance*kpc)**2) # L[erg/s/erg] -> F[mJy]

### Define some useful functions

In [49]:
def safe_log(x):
    return np.where(x > 0, np.log(x), -300)

#### Obtenemos la emisividad y absorcion para cada z

In [50]:
def get_j_alpha(nu, delta, filename='./../Code/Em_Abs.dat'):
    # Read the data from the file
    data = pd.read_csv(filename, delim_whitespace=True, header=None, names=["energy", "emisivity", "absorption", "z_value"])
    
    # Find the energy associated with the given frequency nu
    energy_nu = h * nu * 1e9  # Convert frequency to energy
    
    # Filter the data for each z_value
    filtered_data = []
    for z in data["z_value"].unique():
        z_data = data[data["z_value"] == z]
        closest_index = (np.abs(z_data["energy"] - energy_nu/delta[z-1])).idxmin()
        filtered_data.append(z_data.loc[closest_index, ["energy", "emisivity", "absorption"]])

    # Combine the filtered data into a DataFrame
    filtered_data = pd.DataFrame(filtered_data)
    return filtered_data['emisivity'], filtered_data['absorption']

#### Obtenemos los valores de las cantidades que varian con z

In [51]:
def get_z_rz(filename='./../Code/quant_FF.dat',filename2='./../Code/quant_FF_norm.dat'):
    """
    Load and return specific columns from the given file.

    Parameters:
        filename (str): Path to the file containing the data.

    Returns:
        tuple: Arrays of z, B, r_out, r_in, and delta_obs values.
    """
    return np.loadtxt(filename, usecols=(0, 1, 2, 3, 5), unpack=True)

#### Funcion que calcula l(x,z)

In [52]:
def crossed_length(x, R_in, R_out):
        if abs(x) < abs(R_out) and abs(x) > abs(R_in):
                l = 2.0 * np.sqrt(R_out**2 - x**2)
        elif abs(x) <= abs(R_in):
                l = 2.0 * (np.sqrt(R_out**2 - x**2) - np.sqrt(abs(R_in**2 - x**2)))
        else:
                l = 0.0
        return l

#### Temperatura de Brillo

In [53]:
def Omega_beam(theta1, theta2):
    '''Beam solid angle in terms of the theta_HPBW (theta1 x theta2)
    Use the equation here: https://science.nrao.edu/facilities/vla/proposing/TBconv'''
    return np.pi * theta1 * theta2 / (4 * np.log(2))

def brightness_temperature(I, nu_hz):
    """
    Calculate the brightness temperature for a given intensity and frequency.

    Parameters:
    I (float or numpy.ndarray): Specific intensity [erg/s/cm^2/Hz].
    nu (float): Frequency [Hz].

    Returns:
    float or numpy.ndarray: Brightness temperature [K].
    """
    nu_hz *= 1e9 # Convert GHz to Hz
    

    return I/Omega_beam(0.0168*1e-3,0.0259*1e-3)#(c**2 * I) / (2 * k_B * nu_hz**2*Omega_beam(np.deg2rad(23 * 1e-3 / 3600),np.deg2rad(45 * 1e-3 / 3600))) #[K]

#### Calculamos el valor de la intensidad I(x,z)

In [192]:
#Esta interpola las funciones de z
def Intensity_import_interp(x_value, nu, z_value):
    '''
    Function that returns the intensity for a given frequency at a given range of z, determined by the length of the input array.
    The function uses the values of absoption and emissivity imported from the function get_j_alpha that are closest to the specified energy
    (given by nu) to calculate the intensity based on the input parameters.
    
    This function interpolates for any given value of z'''

    # Retrieve data
    if not hasattr(Intensity_import_interp, "_cached_data"):
        z, B, r_out, r_in, delta_obs = get_z_rz()
        delta_obs = np.sqrt(delta_obs)
        j, alpha = get_j_alpha(nu, delta_obs)
        Intensity_import_interp._cached_data = (z, B, r_out, r_in, delta_obs, j, alpha)
    else:
        z, B, r_out, r_in, delta_obs, j, alpha = Intensity_import_interp._cached_data
    j=j.to_numpy()
    alpha=alpha.to_numpy()
    # Precompute constants

    # Interpolations
    I_z_values = [delta_obs[idx]**3 * j[idx] / alpha[idx] for idx in range(len(z))]
    term_exp_values = [alpha[idx] / delta_obs[idx] for idx in range(len(z))]
    # Remove NaN values from I_z_values and term_exp_values
    valid_indices = ~np.isnan(I_z_values) & ~np.isnan(term_exp_values)
    I_z_values = np.array(I_z_values)[valid_indices]
    term_exp_values = np.array(term_exp_values)[valid_indices]
    z2 = np.array(z)[valid_indices]

    I_z = interp1d(safe_log(z2), safe_log(I_z_values), bounds_error=False, fill_value='extrapolate')
    term_exp = interp1d(safe_log(z2), safe_log(term_exp_values), bounds_error=False, fill_value='extrapolate')
    r_out_interp = interp1d(safe_log(z), safe_log(r_out), bounds_error=False, fill_value='extrapolate')
    r_in_interp = interp1d(safe_log(z), safe_log(r_in), bounds_error=False, fill_value='extrapolate')

    # Compute intensity
    intensity_result = np.zeros(len(x_value))
    r_avg=0.5*(np.exp(r_out_interp(safe_log(z_value)))+np.exp(r_in_interp(safe_log(z_value))))
    tau_avg=crossed_length(r_avg, np.exp(r_in_interp(safe_log(z_value))), np.exp(r_out_interp(safe_log(z_value)))) * np.exp(term_exp(safe_log(z_value))) / np.sin(incl)
    for j, x in enumerate(x_value):
        tau = crossed_length(x, np.exp(r_in_interp(safe_log(z_value))), np.exp(r_out_interp(safe_log(z_value)))) * np.exp(term_exp(safe_log(z_value))) / np.sin(incl)
        intensity_result[j] = np.exp(I_z(safe_log(z_value))) * (-np.expm1(-tau)) #(1+tau)# opt thin #[erg/cm^2/s/erg/sr]
    

    return intensity_result*h, tau_avg #[erg/cm^2/s/Hz/sr]



In [193]:
z, _, r_out, r_in, _ = get_z_rz()
z = np.linspace(z.min(), z.max(), 200)
x = np.linspace(-r_out.max(), r_out.max(), 200)
#x = np.linspace(-r_out.max(), r_out.max(), 200)
# Create a meshgrid for plotting
X_m, Z_m = np.meshgrid(x, z)
I = np.zeros((len(z), len(x)))
tau_avg= np.zeros(len(z))
for i, z_value in enumerate(z):
    I[i], tau_avg[i]=Intensity_import_interp(x, 230.0, z_value) #[erg/cm^2/s/Hz/sr]

/tmp/ipykernel_18864/3843225759.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(filename, delim_whitespace=True, header=None, names=["energy", "emisivity", "absorption", "z_value"])
/tmp/ipykernel_18864/1768712390.py:23: RuntimeWarning: invalid value encountered in scalar divide
  I_z_values = [delta_obs[idx]**3 * j[idx] / alpha[idx] for idx in range(len(z))]


In [194]:
if hasattr(Intensity_import_interp, "_cached_data"):
    del Intensity_import_interp._cached_data

In [195]:
# # Read the CSV file "EHT_OBS(RvZ)" and store its columns in two arrays
# obs_data = pd.read_csv('./EHT_OBS(RvZ).csv')
# obs_z = obs_data.iloc[:, 0].to_numpy()
# obs_r = obs_data.iloc[:, 1].to_numpy()

In [196]:
# z, B, r_out, r_in, delta_obs = get_z_rz()
# plt.figure(figsize=(10, 6))
# # Plot the data
# plt.plot(z*convert*1e3, 2*r_in*convert*1e3, label=fr'$R_{{in,0}}$ = {r_in.min()/R_g:.2f} $R_g$')
# plt.scatter(obs_z, obs_r, color='red', marker='o', label='EHT OBS')
# # Set logarithmic scale
# plt.xscale('log')
# plt.yscale('log')
# plt.xlim(15, 300)
# plt.ylim(15, 70)
# plt.xticks([20,40,60,80,100,200], [20,40,60,80,100,200])
# plt.yticks([20,40,60], [20,40,60])
# plt.title(fr'Width vs Z, $Z_0$={z.min()/R_g:.2f} $R_g$')
# # Add labels, title, and legend
# plt.ylabel(r'2$\cdot R_{in}$  $[\mu as]$')
# plt.xlabel(r'$Z \cdot \sin ( \theta) [\mu as]$')
# plt.legend()
# plt.grid()
# id=input('Grafico Z vs R_in, ingrese id: ')
# plt.savefig(fr'./plots/Z_vs_R_in({id}).png', dpi=300)
# plt.show()

In [197]:
def Convolve_map(x_map, y_map, F_map, sigma_x, sigma_y):
    # Calculate map boundaries
    y_min_c = y_map[0] - 5. * sigma_y
    y_max_c = y_map[-1] + 5. * sigma_y
    x_min_c = x_map[0] - 4. * sigma_x
    x_max_c = x_map[-1] + 4. * sigma_x

    ancho_x = x_max_c - x_min_c
    ancho_y = y_max_c - y_min_c

    Dx_c = f_Ny * sigma_x
    Dy_c = f_Ny * sigma_y

    Dx_c_rad = np.deg2rad(Dx_c * 1e-3 / 3600)
    Dy_c_rad = np.deg2rad(Dy_c * 1e-3 / 3600)

    F_map = F_map * Dx_c_rad * Dy_c_rad

    Nx_c = int(ancho_x / Dx_c)
    Ny_c = int(ancho_y / Dy_c)

    x_c = np.linspace(x_min_c, x_max_c, Nx_c)
    y_c = np.linspace(y_min_c, y_max_c, Ny_c)

    F_map_c = np.zeros((Nx_c, Ny_c))

    # Prepare meshgrids for vectorized computation
    X_map, Y_map = np.meshgrid(x_map, y_map, indexing='ij')

    w_x = 0.5 / sigma_x ** 2
    w_y = 0.5 / sigma_y ** 2

    for i in range(Nx_c):
        for j in range(Ny_c):
            gaussian = np.exp(-((X_map - x_c[i]) ** 2 * w_x + (Y_map - y_c[j]) ** 2 * w_y))
            F_map_c[i, j] = np.sum(F_map * gaussian)

    S_pre = np.sum(F_map)
    A_beam = 2 * np.pi * sigma_x * sigma_y
    A_cell = Dx_c * Dy_c
    mJy_beam = A_cell / A_beam
    S_post = np.sum(F_map_c) * mJy_beam

    # Uncomment for debugging
    # print('Pre-conv: S(GHz) =', S_pre, 'mJy')
    # print('Post-conv: S(GHz) =', S_post, 'mJy')
    # print('post/pre =', S_post / S_pre)

    return x_c, y_c, F_map_c


In [198]:
FWHM_x, FWHM_y= 0.0168, 0.0259 #FWHM in mas
sigma_x, sigma_y =FWHM_x/(2*np.sqrt(2*np.log(2))), FWHM_y/(2*np.sqrt(2*np.log(2))) 
X_c, Y_c, I_c = Convolve_map(X_m[0]*convert, Z_m.T[0]*np.sin(incl)*convert, I.T*1e26, sigma_x, sigma_y)

In [199]:
# from matplotlib.patches import Ellipse
# # nice_fonts = {
# #         # Use LaTeX to write all text
# #         "text.usetex": False,
# #         "font.family": 'serif',
# #         # Use 10pt font in plots, to match 10pt font in document
# #         "axes.labelsize": 16,
# #         "font.size": 16,
# #         "axes.linewidth":1,
# #         "axes.titlesize":20,
# #         # Make the legend/label fonts a little smaller
# #         "legend.fontsize": 10,
# #         "xtick.labelsize": 14,
# #         "ytick.labelsize": 14,
# # }
# # matplotlib.rcParams.update(nice_fonts)

# #plt.clf()
# with plt.style.context(('classic')):
#     fig, ax = plt.subplots(1, 1, figsize=(7.8, 6.6)) # proportion: 2.6/2.2
#     #fig, ax = plt.subplots(1, 1) 
#     ax.set_xlabel(r"$z\cdot\sin(\theta)~{\rm [mas]}$")
#     ax.set_ylabel(r"$x~{\rm [mas]}$")
#     ax.set_xlim([-0.05, .60])
#     ax.set_ylim([-0.3, 0.3])
#     palette = plt.cm.afmhot
#     #cmap.set_label(r"$S_\nu$ [mJy beam$^{-1}$]")
#     ticks = [20,40,60,80,100]#[0.01, 0.05,  0.1,  0.15]
#     norm = matplotlib.colors.Normalize(vmin=0, vmax=20)
#     CS_conv = ax.contourf(Y_c, X_c, I_c, cmap=palette, norm=norm, levels=np.linspace(0,60, 500))
#     # ticks = [0.001,0.01,0.1,1,10,100,1000,10000,1e5]#[0.01, 0.05,  0.1,  0.15]
#     # norm = matplotlib.colors.LogNorm(vmin=5e-1, vmax=I_c.max())
#     # CS_conv = ax.contourf(Y_c, X_c, I_c, cmap=palette, norm=norm, levels=np.geomspace(0.01,1e3, 500))

#     # Add contours representing 1, 2, and 3 sigma levels
#     sigma_levels = [3,2,1]
#     obs_cont = [np.exp(-0.5 * (level**2)) * I_c.max() for level in sigma_levels]
#     plot_cont = ax.contour(Y_c, X_c, I_c, linewidths=1, levels=obs_cont, extend="both", colors='dimgrey')

#     obs_cont = np.geomspace(0.01, 1e3, 10)
#     plot_cont = ax.contour(Y_c, X_c, I_c, linewidths=1, levels=obs_cont, extend="both", colors='dimgrey', linestyles='--')

#     el = Ellipse((0.05,0.1), FWHM_y, FWHM_x, color='grey')
#     ax.add_patch(el)

#     ax.grid()
#     plt.tight_layout()
#     fig.colorbar( CS_conv, ticks=ticks ,label=r'$S_\nu(x,z)~[mJy]$')
#     plt.savefig(fr'./plots/Mapa_Intensidad_convolucionado.png', dpi=300)

In [200]:
T_brillo=brightness_temperature(I_c, 230.0)

import os

filename = "T_brillo_output.txt"
mode = "a" if os.path.exists(filename) else "w"

with open(filename, mode) as f:
    if mode == "a":
        f.write("\n")
    np.savetxt(f, T_brillo)

In [201]:
# import numpy as np

# def read_T_brillo_arrays(filename="T_brillo_output.txt"):
#     arrays = []
#     with open(filename, "r") as f:
#         content = f.read()
#         blocks = [block for block in content.strip().split('\n\n') if block.strip()]
#         for block in blocks:
#             arr = np.loadtxt(block.splitlines())
#             arrays.append(arr)
#     return arrays

# T_brillo_arrays = read_T_brillo_arrays("T_brillo_output.txt")
# print(len(T_brillo_arrays))

In [202]:
# T_brillo_1=T_brillo_arrays[0]
# T_brillo_2=T_brillo_arrays[1]
# T_brillo_3=T_brillo_arrays[2]
# T_brillo_4=T_brillo_arrays[3]
# T_brillo_5=T_brillo_arrays[4]

In [203]:
T_b=T_brillo
# T_b

In [204]:
# with plt.style.context(('classic')):
#     fig, ax = plt.subplots(1, 1, figsize=(7.8, 6.6)) # proportion: 2.6/2.2
#     #fig, ax = plt.subplots(1, 1) 
#     ax.set_xlabel(r"$z\cdot\sin(\theta)~{\rm [mas]}$")
#     ax.set_ylabel(r"$x~{\rm [mas]}$")
#     ax.set_xlim([-0.05, 0.55])
#     ax.set_ylim([-0.3, 0.3])
#     palette = plt.cm.afmhot
#     #cmap.set_label(r"$S_\nu$ [mJy beam$^{-1}$]")
#     ticks = [0.5,1,1.5,2,2.5,3,3.5]  # Adjusted ticks for sqrt(T_b) in Gigakelvin
#     norm = matplotlib.colors.Normalize(vmin=0.5, vmax=5)
#     CS_conv = ax.contourf(Y_c, X_c, np.sqrt(T_b/1e9), cmap=palette, norm=norm, levels=np.linspace(0,4.5 , 700))

#     #Agregar contornos a (1, 1.4, 2, 2.8...)*3 mJy beam^-1 --> son 19? aprox
#     obs_cont = np.array([-1,-0.5,0,0.5,1,2])
#     plot_cont = ax.contour(Y_c, X_c, np.sqrt(T_b/1e9), linewidths=1, levels=obs_cont, extend="both", colors=['dimgrey'])

#     el = Ellipse((0.0, 0.075), sigma_y, sigma_x, color='grey')
#     ax.add_patch(el)

#     ax.grid()
#     plt.tight_layout()
#     fig.colorbar( CS_conv, ticks=ticks ,label=r'$\sqrt{T_b~[10^9~K]}$')
    
#     id=input('Grafico T_b, ingrese id: ')
#     plt.savefig(fr'./plots/SQRT_T_brightness_(230GHz){id}.png', dpi=300)

In [205]:
# # Read the CSV file "EHT_OBS(RvZ)" and store its columns in two arrays
# obs_data = pd.read_csv('./EHT_OBS(TBvZ).csv')
# obs_z = obs_data.iloc[:, 0].to_numpy()
# obs_TB = obs_data.iloc[:, 1].to_numpy()

In [206]:
# # Take the highest value along the X_c axis for each Y_c (i.e., max over rows)
# T_b_df = pd.DataFrame(T_b/1e9, index=X_c, columns=Y_c)
# T_b_df_plus=T_b_df[T_b_df.index > 0] #Promedio con valores de x positivos
# T_b_df_minus=T_b_df[T_b_df.index < 0] #Promedio con valores de x positivos
# # T_b_mean_values = T_b_df_plus.mean(axis=0)
# # T_b_mean_values_minus = T_b_df_minus.mean(axis=0)
# T_b_mean_values = T_b_df_plus.max(axis=0)
# T_b_mean_values_minus = T_b_df_minus.max(axis=0)

# plt.figure(figsize=(10, 6))
# plt.plot(Y_c*1e3, T_b_mean_values, label='$x>0$')
# plt.scatter(obs_z, obs_TB, color='red', marker='o', label='EHT OBS')
# plt.plot(Y_c*1e3, -T_b_mean_values_minus, label='$x<0$')
# plt.xlabel(fr'$Z~[\mu as]$')
# plt.ylabel(r'Temperatura de brillo $[10^9~K]$')
# plt.xlim(-50, 350)
# plt.legend()
# plt.grid()

# # Set y-ticks to absolute values and color positive as blue, negative as orange
# ax = plt.gca()
# y_ticks = ax.get_yticks()
# tick_labels = []
# tick_colors = []
# for y in y_ticks:
#     tick_labels.append(f"{abs(y):.0f}")
#     if y >= 0:
#         tick_colors.append('blue')
#     else:
#         tick_colors.append('orange')
# ax.set_yticklabels(tick_labels)
# for tick_label, color in zip(ax.get_yticklabels(), tick_colors):
#     tick_label.set_color(color)

# id=input('Grafico T_b vs Z, ingrese id: ')
# plt.savefig(fr'./plots/T_b_vs_Z_({id}).png', dpi=300)
# plt.show()


In [207]:
# # Take the highest value along the X_c axis for each Y_c (i.e., max over rows)
# T_b1_df = pd.DataFrame(T_brillo_1/1e9, index=X_c, columns=Y_c)
# T_b1_df_plus=T_b1_df[T_b1_df.index > 0] #Promedio con valores de x positivos
# T_b2_df = pd.DataFrame(T_brillo_2/1e9, index=X_c, columns=Y_c)
# T_b2_df_plus=T_b2_df[T_b2_df.index > 0] #Promedio con valores de x positivos
# T_b3_df = pd.DataFrame(T_brillo_3/1e9, index=X_c, columns=Y_c)
# T_b3_df_plus=T_b3_df[T_b3_df.index > 0] #Promedio con valores de x positivos
# T_b4_df = pd.DataFrame(T_brillo_4/1e9, index=X_c, columns=Y_c)
# T_b4_df_plus=T_b4_df[T_b4_df.index > 0] #Promedio con valores de x positivos
# T_b5_df = pd.DataFrame(T_brillo_5/1e9, index=X_c, columns=Y_c)
# T_b5_df_plus=T_b5_df[T_b5_df.index > 0] #Promedio con valores de x positivos
# # T_b_df_minus=T_b_df[T_b_df.index < 0] #Promedio con valores de x positivos
# T_b1_mean_values = T_b1_df_plus.max(axis=0)
# T_b2_mean_values = T_b2_df_plus.max(axis=0)
# T_b3_mean_values = T_b3_df_plus.max(axis=0)
# T_b4_mean_values = T_b4_df_plus.max(axis=0)
# T_b5_mean_values = T_b5_df_plus.max(axis=0)
# # T_b_mean_values_minus = T_b_df_minus.mean(axis=0)
# # T_b_mean_values = T_b_df_plus.max(axis=0)
# # T_b_mean_values_minus = T_b_df_minus.max(axis=0)

# plt.figure(figsize=(10, 6))
# plt.plot(Y_c*1e3, T_b1_mean_values, linestyle='-', label='$-0.01$')
# plt.plot(Y_c*1e3, T_b2_mean_values,linestyle='--', label='$-0.1$')
# plt.plot(Y_c*1e3, T_b3_mean_values,linestyle=':', label='$-0.5$')
# plt.plot(Y_c*1e3, T_b4_mean_values,linestyle='-.', label='$z_0=4R_g$')
# plt.plot(Y_c*1e3, T_b5_mean_values, linestyle='-', label='$z_0=6R_g$')
# plt.axvline(x=2*R_g*convert*1e3, color='red', linestyle='-.', label='$2R_g$')
# #plt.plot(Y_c*1e3, -T_b_mean_values_minus, label='$x<0$')
# plt.xlabel(fr'$Z~[\mu as]$')
# plt.ylabel(r'Temperatura de brillo $[10^9~K]$')
# plt.xlim(-50, 350)
# plt.legend()
# plt.grid()

# # Set y-ticks to absolute values and color positive as blue, negative as orange
# ax = plt.gca()
# y_ticks = ax.get_yticks()
# tick_labels = []
# tick_colors = []
# for y in y_ticks:
#     tick_labels.append(f"{abs(y):.0f}")
#     if y >= 0:
#         tick_colors.append('blue')
#     else:
#         tick_colors.append('orange')
# ax.set_yticklabels(tick_labels)
# for tick_label, color in zip(ax.get_yticklabels(), tick_colors):
#     tick_label.set_color(color)

# plt.savefig('./plots/T_b_vs_Z_(mean).png', dpi=300)
# plt.show()
